
# P2 : Variables catégorielles et feature engineering

## 1. Chargement des données et variables catégorielles

In [ ]:
import pandas as pd 
import numpy as np 

train= pd.read_csv("../data/train_final.csv")
cible="Purchased"
cat_cols=["CampaignType","DeviceOS","BrowserCode","GeoZone","EntryChannel","CustomerType"]

## 2. Modalités et taux d'achat par catégorie

In [8]:
for c in cat_cols:
    res=train.groupby(c)[cible].agg(effectif="count",taux_achat="mean")
    res["part"]=res["effectif"]/len(train)
    print("\n",c, " : \n")
    print(res.sort_values("effectif",ascending=False).round(3))



 CampaignType  : 

              effectif  taux_achat   part
CampaignType                             
Regular           3358       0.144  0.478
Seasonal          1602       0.155  0.228
FlashDeal          772       0.166  0.110
Holiday            724       0.177  0.103
Clearance          564       0.161  0.080

 DeviceOS  : 

          effectif  taux_achat   part
DeviceOS                             
2             3721       0.155  0.530
1             1520       0.155  0.217
3              714       0.146  0.102
4              545       0.141  0.078
5              294       0.156  0.042
6              130       0.138  0.019
7               56       0.214  0.008
8               40       0.175  0.006

 BrowserCode  : 

             effectif  taux_achat   part
BrowserCode                             
2                4094       0.155  0.583
1                1537       0.139  0.219
3                 405       0.175  0.058
4                 316       0.171  0.045
5                 235    

## 3. Catégories rares


In [10]:
for c in cat_cols:
    part = train[c].value_counts(normalize=True)    
    print(c, " rares" , list(part[part < 0.01].index))

CampaignType  rares []
DeviceOS  rares [7, 8]
BrowserCode  rares [8, 10, 9, 11, 12, 13]
GeoZone  rares []
EntryChannel  rares [13, 14, 17, 15, 16, 18, 19, 20]
CustomerType  rares ['Other']


## 4. Nouvelles variables et corrélation avec la cible

In [ ]:
def sdiv(a, b):
    return train[a] / train[b].replace(0, np.nan)

nouv = pd.DataFrame({
    "total_pages": train["AdminPages"] + train["InfoPages"] + train["CatalogPages"],
    "total_time": train["AdminTimeSec"] + train["InfoTimeSec"] + train["CatalogTimeSec"],
    "time_per_page_catalog": sdiv("CatalogTimeSec", "CatalogPages"),
    "exit_sur_bounce": sdiv("ExitRate", "BounceRate"),
})
nouv["engagement_par_page"] = train["EngagementValue"] / nouv["total_pages"].replace(0, np.nan)
nouv[cible] = train[cible]
print(nouv.corr()[cible].round(3),"\n")
print(train.select_dtypes("number").corr()[cible].round(3))

total_pages              0.205
total_time               0.185
time_per_page_catalog    0.067
exit_sur_bounce          0.001
engagement_par_page      0.113
Purchased                1.000
Name: Purchased, dtype: float64 

AdminPages         0.103
AdminTimeSec       0.100
InfoPages          0.050
InfoTimeSec        0.039
CatalogPages       0.201
CatalogTimeSec     0.181
BounceRate        -0.132
ExitRate          -0.188
EngagementValue    0.339
PromoProximity     0.009
DeviceOS          -0.001
BrowserCode        0.017
GeoZone           -0.021
EntryChannel      -0.006
Weekend            0.033
Purchased          1.000
Name: Purchased, dtype: float64


Conclusion : EngagementValue est la variable numérique originale qui présente la relation linéaire la plus importante avec Purchased (+0.339). CatalogPages (+0.201) et CatalogTimeSec (+0.181) présentent également des relations positives faibles. En revanche, BounceRate et ExitRate présentent des relations négatives faibles.

EngagementValue est la variable la plus corrélée à l'achat.

total_pages et total_time sont les nouvelles variables les plus intéressantes.

Les variables catégorielles codées numériquement (DeviceOS, BrowserCode, GeoZone, EntryChannel) ont des corrélations proches de zéro. Cela ne signifie pas qu'elles sont inutiles : leur codage numérique ne représente pas nécessairement un ordre réel.

Une corrélation faible ne signifie pas qu'une variable n'apporte aucune information au modèle.